# Vamos aprender a trabalhar com PDF usando o Python

- Regra geral: PDF foi feito justamente para bloquear muita coisa, então não é fácil "brincar" com um pdf
- Mesmo assim, Python tem várias bibliotecas que vão nos ajudar, vamos focar em 2:
    - PyPDF2
    - Tabula
- Ler e extrair informações de um PDF a gente consegue fazer.
- Escrever e Editar, aí já é outra história

### Para os nossos exemplos, vamos avaliar o Release de Resultados do 3º e 4º Trimestre de 2020 da Magazine Luiza

#### 1º Objetivo: Queremos conseguir separar apenas o DRE do Release de Resultados (Página 14) para enviar para a Diretoria, como fazemos?
    - Separar as páginas de um pdf

In [ ]:
import PyPDF2 as pyf

nome = 'MGLU_ER_3T20_POR.pdf'
arquivo_pdf = pyf.PdfReader(nome)

arquivo_pdf.pages

TypeError: 'PdfReader' object is not iterable

#### 2º Objetivo: Com o Release de Resultados já separado página por página, queremos incluir apenas as Páginas de Destaque (Página 1), DRE (Página 14) e Balanço (Página 16).
    - Juntar vários pdfs em 1

### Extra: Para adicionar todas as páginas de 2 pdfs

# Funcionalidades que podem ser úteis:

- Inserir arquivo no meio do outro
- Quero colocar dentro do Resultado do 4T20 os destaques do 3T20 para poder comparar os 2 dentro do mesmo relatório

- Rodar Página

# Trabalhando com Textos e Informações Dentro do PDF

#### 1º Objetivo: Quero identificar como foram as Despesas com Vendas da MGLU
    - Pegar texto da página e identificar onde está essa informação

In [ ]:
posicao_inicial = texto_analisar.find(texto_referencia)
posicao_final = texto_analisar.find('|', posicao_inicial+1)

texto_final = texto_analisar[posicao_inicial:posicao_final]
print(texto_final)

#### 2º Objetivo: Quero analisar o DRE (sem ajuste - Página 5)
    - Para ler tabelas em pdf, use o tabula (é ninja)
    
    - Cuidado 1: Instale o tabula-py (não instale o tabula). Se instalar o tabula errado, desinstale ele, instale o tabula-py, desinstale o tabula-py e instale novamente o tabula-py. Reinicie o kernel do Jupyter após isso
    
    - Cuidado 2: Tem que ter o java instalado no seu computador (depois de instalar, reinicie o computador)

In [ ]:
import tabula
import pandas as pd

tabelas = tabula.read_pdf('MGLU_ER_3T20_POR.pdf', pages=5)
print(type(tabelas))

# como se tem somente 1 tabela, entao pode-se extrair diretamente a unica tabela que se tem
tabelas = tabelas[0]
df_resultado = tabelas
# display(df_resultado)

# excluindo linhas vazias
df_resultado = df_resultado.dropna(how='all', axis=0)

# excluindo colunas vazias
df_resultado = df_resultado.dropna(how='all', axis=1)

#fixando a primeira linha // colcando a partir de 1 // resetando index
df_resultado.columns = df_resultado.iloc[0]
df_resultado = df_resultado.iloc[1:]
df_resultado = df_resultado.reset_index(drop=True)

display(df_resultado)


#### 3º Objetivo: Quero analisar o Capital de Giro e os Investimentos (ambas as tabelas na página 12)
    - Páginas com mais de 1 tabela

In [ ]:
tabelas = tabula.read_pdf('MGLU_ER_3T20_POR.pdf', pages=12)
print(len(tabelas))
for tabela in tabelas:
    tabela = tabela.dropna(how='all', axis=0)
    display(tabela)

In [ ]:
df_capitalgiro = tabelas[0]
tabela = tabela.dropna(how='all', axis=0)
display(tabela)
 

#### O que fazer quando o tabula não consegue ler alguma linha da tabela? Como o cabeçalho, no nosso caso?

# Outro método que pode ser útil algum dia: Captar Imagem em um pdf
    - biblioteca pikepdf

In [ ]:
from pikepdf import Pdf, PdfImage

arquivo = Pdf.open('MGLU_ER_3T20_POR.pdf')

for pagina in arquivo.pages[:1]:
    imagens = pagina.images.items()
    print(imagens)

print('\n')
for pagina in arquivo.pages[:1]:
    for nome_imagem, imagem in pagina.images.items():
        print(type(imagem))
        print(type(nome_imagem))
        # imagem_salvar = PdfImage(imagem)
        # imagem_salvar.extract_to(fileprefix=f'imagens/{nome_imagem}')

# INCOMPLETO!!!

In [ ]:
from pikepdf import Pdf, PdfImage

arquivo = Pdf.open('MGLU_ER_3T20_POR.pdf')

for pagina in arquivo.pages[:1]:
    imagens = pagina.images.items()
    print(imagens)

In [ ]:
import pikepdf
print(pikepdf.__version__)

In [ ]:
import pikepdf
print(pikepdf.__version__)

In [ ]:
from pikepdf import Pdf, PdfImage

arquivo = Pdf.open('MGLU_ER_3T20_POR.pdf')

for pagina in arquivo.pages[:1]:
    print(pagina.images.items())
    
    for nome, imagem in pagina.images.items():
        print(nome)

In [ ]:
from pikepdf import Pdf, PdfImage

arquivo = Pdf.open('MGLU_ER_3T20_POR.pdf')

for pagina in arquivo.pages[:1]:
    for nome, imagem in pagina.images.items():
        imagem_salvar = PdfImage(imagem)
        imagem_salvar.extract_to(fileprefix=f'imagens/{nome}')

In [ ]:
import pandas as pd
import requests
import io


url = 'https://portalweb.cooxupe.com.br:9080/portal/precohistoricocafe_2.jsp;jsessionid=29E6BF0EAAE6CF9FB927E92F58DF60A4?d-3496238-e=2&6578706f7274=1'
conteudo_url = requests.get(url).content 
arquivo_url = io.StringIO(conteudo_url.decode('latin1'))
cafe_df = pd.read_csv(arquivo_url, sep='\t')
display(cafe_df)

# Substituir texto no pdf tipo contrato

- Não recomendo fazer diretamente pelo Python. Realmente do que vi a melhor opção me parece o Word fazer isso
- Caso precise automatizar, automatize o processo fazendo ele pelo Word
- Quem quiser MUITO fazer isso pelo Python, tem um link aqui que vai te ajudar de uma solução que achei que funciona. Tem seus bugs/cuidados especiais, mas funciona: https://pdf.co/samples/pdf-co-web-api-replace-text-from-pdf-python-replace-text-from-url